# WhatsApp Chat Analyzer

**Exploratory analysis of an exported WhatsApp group chat using Python, Pandas and Matplotlib.**

## Overview
This notebook turns a raw WhatsApp chat export (`.txt`) into a structured dataset and extracts activity insights: who talks the most, when the chat is most active, which words and emojis are used most often, and how activity changes over time. The same logic powers the Streamlit web app built from this project.

## Contents
1. Imports
2. Loading the chat file
3. Preprocessing (regex parsing, date features)
4. Top statistics
5. Most busy users
6. Word analysis
7. Emoji analysis
8. Timelines
9. Activity maps
10. Weekly activity heatmap
11. Summary and next steps

## Tech stack
Python, Pandas, Regex (`re`), Matplotlib, Seaborn, WordCloud, URLExtract, Emoji

## About the data
- Source: WhatsApp group chat exported **without media** (Android format: `dd/mm/yy, hh:mm am/pm - Name: message`)
- Resulting columns: `date`, `user`, `message`, plus derived `year`, `month`, `month_num`, `day`, `day_name`, `hour`, `minute`, `only_date`, `period`
- Lines without a sender (joins, encryption notice, etc.) are labelled `group_notification` and excluded from most analyses
- The chat file contains private conversations, so it is **not included in the repository**

## 1. Imports

In [ ]:
import re
import pandas as pd

## 2. Loading the chat file
Read the exported chat as a UTF-8 text file into a single string.

In [ ]:
f = open("WhatsApp Chat with 10th grade friendship .txt",'r',encoding='utf-8')

In [ ]:
data = f.read()

In [ ]:
print(data)

## 3. Preprocessing
The raw export is one long text. Each message starts with a timestamp, so a regex is used to split the text into dates and messages.

### 3.1 Split messages and dates with regex

In [ ]:
pattern = r'\d{1,2}/\d{1,2}/\d{2},\s\d{1,2}:\d{2}\s(?:am|pm)\s-\s'

In [ ]:
messages = re.split(pattern,data)[1:]
messages

In [ ]:
dates = re.findall(pattern,data)
dates

### 3.2 Build the DataFrame and parse dates
Combine messages and timestamps into a DataFrame and convert the timestamps to `datetime` (12-hour format with am/pm).

In [ ]:
df = pd.DataFrame({'user_message':messages, 'message_date':dates})
# convert message_date type
df['message_date'] = pd.to_datetime(df['message_date'], format= '%d/%m/%y, %I:%M %p - ')

df.rename(columns={'message_date': 'date'}, inplace=True)

df.head()

In [ ]:
df.shape

### 3.3 Separate users and messages
Split each entry into sender and message text. Entries with no sender are labelled `group_notification`.

In [ ]:
# seperate users and messages
users = []
messages = []
for message in df['user_message']:
    entry = re.split(r'([\w\W]+?):\s',message)
    if entry[1:]: #user name
        users.append(entry[1])
        messages.append(entry[2])
    else:
        users.append('group_notification')
        messages.append(entry[0])

df['user'] = users
df['message'] = messages
df.drop(columns=['user_message'], inplace=True)

df.head()

### 3.4 Date features
Extract year, month, day, hour and minute for time-based analysis.

In [ ]:
df['year'] = df['date'].dt.year

In [ ]:
df.head()

In [ ]:
df['month'] = df['date'].dt.month_name()
df.head()

In [ ]:
df['day'] = df['date'].dt.day
df.head()

In [ ]:
df['hour'] = df['date'].dt.hour
df.head()

In [ ]:
df['minute'] = df['date'].dt.minute
df.head()

## 4. Top statistics
Basic counts for the whole chat: total words, media messages and links shared.

### 4.1 Total words and media messages

In [ ]:
words = []
for message in df['message']:
    words.extend(message.split())

In [ ]:
len(words)

In [ ]:
df[df['message'] == '<Media omitted>\n']

### 4.2 Links shared
URLs are detected with the `urlextract` library.

In [ ]:
pip install urlextract

In [ ]:
from urlextract import URLExtract
extractor = URLExtract()
urls = extractor.find_urls('lets check www.gmail.com')
urls

In [ ]:
links = []
for message in df['message']:
    links.extend(extractor.find_urls(message))


In [ ]:
len(links)

## 5. Most busy users
The five users with the most messages, shown as a bar chart and as a percentage share of all messages.

In [ ]:
x = df['user'].value_counts().head()

In [ ]:
pip install matplotlib

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
name = x.index
count = x.values

In [ ]:
plt.bar(name,count)
plt.xticks(rotation='vertical')
plt.show()

In [ ]:
new_df = round((df['user'].value_counts() / df.shape[0]) * 100, 2).reset_index()
new_df.columns = ['name', 'percent']

## 6. Word analysis
Find the most frequently used words after removing noise.

### 6.1 Remove group notifications and media messages

In [ ]:
temp = df[df['user'] != 'group_notification']
temp = temp[temp['message'] != '<Media omitted>\n']


In [ ]:
#remove group messages - done
#remove media omitted message - done
#remove stop words

### 6.2 Remove stop words and find the most common words
Stop words (English, Hindi and Marathi written in Roman script) are loaded from a text file and filtered out.

In [ ]:
f = open('stop_hinglish_marathi.txt','r')
stop_words = f.read()
print(stop_words)

In [ ]:
words = []
for message in temp['message']:
    for word in message.lower().split():
        if word not in stop_words:
            words.append(word)


In [ ]:
from collections import Counter
pd.DataFrame(Counter(words).most_common(20))

## 7. Emoji analysis
Count every emoji used in the chat and rank them by frequency.

In [ ]:
pip install emoji

In [ ]:
import emoji
emojis = []
for message in df['message']:
    emojis.extend([e['emoji'] for e in emoji.emoji_list(message)])
print(len(emojis))
print(emojis[:10])

In [ ]:
emoji_df = pd.DataFrame(Counter(emojis).most_common(), columns=['emoji', 'count'])
emoji_df.head(10)

## 8. Timelines
How the number of messages changes over time.

### 8.1 Monthly timeline

In [ ]:
df['month_num'] = df['date'].dt.month
df

In [ ]:
timeline = df.groupby(['year','month_num','month']).count()['message'].reset_index()

In [ ]:
time = []
for i in range(timeline.shape[0]):
    time.append(timeline['month'][i] + '-' + str(timeline['year'][i]))

In [ ]:
time

In [ ]:
timeline['time'] = time

In [ ]:
timeline

In [ ]:
plt.plot(timeline['time'],timeline['message'])
plt.xticks(rotation='vertical')
plt.show()

### 8.2 Daily timeline

In [ ]:
df['only_date'] = df['date'].dt.date

In [ ]:
daily_timeline = df.groupby('only_date').count()['message'].reset_index()

In [ ]:
plt.figure(figsize=(18,12))
plt.plot(daily_timeline['only_date'],daily_timeline['message'])

## 9. Activity maps
Which days and hours the chat is most active.

### 9.1 Busiest day

In [ ]:
df['day_name'] = df['date'].dt.day_name()


In [ ]:
df['day_name'].value_counts()

### 9.2 Hour period
Each hour is converted into a one-hour slot label (for example `21-22`), zero-padded so the slots sort correctly.

In [ ]:
period = []
for hour in df['hour']:
    period.append(f"{hour:02d}-{(hour + 1) % 24:02d}")

df['period'] = period

In [ ]:
df['period'] = period

## 10. Weekly activity heatmap
Messages by day of the week and hour of the day. Rows are ordered Monday to Sunday and columns from `00-01` to `23-00`.

In [ ]:
pip install seaborn

In [ ]:
import seaborn as sns

In [ ]:
days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
periods = [f"{h:02d}-{(h + 1) % 24:02d}" for h in range(24)]

user_heatmap = df.pivot_table(index='day_name', columns='period',
                              values='message', aggfunc='count').fillna(0)

user_heatmap = user_heatmap.reindex(index=days, columns=periods, fill_value=0)

In [ ]:
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(user_heatmap, ax=ax)
plt.xticks(rotation=45)
plt.show()

## 11. Summary and next steps

**What this notebook does**
- Parses a raw WhatsApp export into a clean, analysis-ready DataFrame
- Computes message, word, media and link statistics
- Visualizes user activity, monthly and daily timelines, busiest days and hours
- Extracts common words (with stop-word removal) and emoji usage

**Possible next steps**
- Support Devanagari text in word clouds and word counts
- Support other chat exports (Telegram, Instagram)
- Add sentiment analysis and response-time analysis
- Move the logic into `preprocessor.py` and `helper.py` and serve it through the Streamlit app